<a href="https://colab.research.google.com/github/Luisinho-31/data-analytics-pharma/blob/main/reporte_clinica_ml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Reporte Automatizado de Clinica Privada con ML
## Analisis Operativo + Modelo de Propension a Enfermedades Cronicas

> **Problema que resuelve:** Una clinica privada necesita dos cosas: visibilidad operativa
> (consultas, ingresos, ocupacion) y capacidad de anticipar que pacientes tienen mayor
> riesgo de desarrollar enfermedades cronicas como diabetes, hipertension u obesidad.
>
> Este reporte combina ambas en un PDF automatico generado con Python.

**Stack:** Python · Pandas · Scikit-learn (Random Forest) · Matplotlib · FPDF2  
**Autor:** Luis Fernando Castillo Garcia | [LinkedIn](https://www.linkedin.com/in/luis-fernando-castillo-garcia) | [GitHub](https://github.com/Luisinho-31)


## 0. Instalacion de dependencias

In [ ]:
!pip install fpdf2 matplotlib pandas numpy scikit-learn --quiet
!pip install fpdf2 --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.0/81.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 13.7 MB/s eta 0:00:00


## 1. Generacion de datos

> Simulamos datos de una clinica privada: consultas, medicos, especialidades y
> perfil clinico de pacientes. En un caso real vendrian de un sistema de gestion o Excel.


In [ ]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
import os
import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)
random.seed(42)

# Catalogo de especialidades y medicos
especialidades = ["Medicina General","Cardiologia","Endocrinologia","Nutricion","Pediatria","Ginecologia"]
medicos = {
    "Medicina General":  ["Dr. Garcia","Dra. Lopez","Dr. Martinez"],
    "Cardiologia":       ["Dr. Hernandez","Dra. Torres"],
    "Endocrinologia":    ["Dra. Ramirez","Dr. Flores"],
    "Nutricion":         ["Dra. Cruz","Dr. Reyes"],
    "Pediatria":         ["Dra. Morales","Dr. Jimenez"],
    "Ginecologia":       ["Dra. Vargas","Dra. Castillo"],
}
precio_consulta = {
    "Medicina General": 500, "Cardiologia": 1200,
    "Endocrinologia": 1100, "Nutricion": 700,
    "Pediatria": 600, "Ginecologia": 900,
}
zonas    = ["Norte","Sur","Este","Oeste","Centro"]
diagnosticos_previos = ["Ninguno","Ninguno","Ninguno","Diabetes","Hipertension",
                        "Obesidad","Diabetes y Hipertension","Prediabetes","Sobrepeso"]

# Generar consultas ultimos 6 meses
fechas = pd.date_range(end=datetime.today(), periods=180, freq='D')
consultas = []
for fecha in fechas:
    n = random.randint(8, 25)
    for _ in range(n):
        esp = random.choice(especialidades)
        med = random.choice(medicos[esp])
        edad = random.randint(18, 80)
        genero = random.choice(["M","F","F","F"])
        imc = round(random.gauss(27.5, 5), 1)
        imc = max(16, min(45, imc))
        pas = random.randint(100, 180)  # presion sistolica
        pad = random.randint(60, 110)   # presion diastolica
        glucosa = random.randint(70, 200)
        diag_previo = random.choice(diagnosticos_previos)
        consultas.append({
            "fecha": fecha,
            "especialidad": esp,
            "medico": med,
            "paciente_id": f"P{random.randint(1000,9999):04d}",
            "edad": edad,
            "genero": genero,
            "zona": random.choice(zonas),
            "imc": imc,
            "presion_sistolica": pas,
            "presion_diastolica": pad,
            "glucosa": glucosa,
            "diagnostico_previo": diag_previo,
            "es_nuevo": random.choice([True,False,False,False]),
            "ingreso": precio_consulta[esp],
        })

df = pd.DataFrame(consultas)
os.makedirs("data", exist_ok=True)
df.to_csv("data/consultas_clinica.csv", index=False)
print(f"Datos generados: {len(df):,} consultas | {df['paciente_id'].nunique():,} pacientes unicos")
df.head(7)

Datos generados: 2,991 consultas | 2,564 pacientes unicos


,fecha,especialidad,medico,paciente_id,edad,genero,zona,imc,presion_sistolica,presion_diastolica,glucosa,diagnostico_previo,es_nuevo,ingreso
0,2025-12-09 03:31:46.404501,Medicina General,Dr. Martinez,P1488,35,F,Norte,28.9,169,65,178,Ninguno,False,500
1,2025-12-09 03:31:46.404501,Cardiologia,Dr. Hernandez,P5557,53,F,Norte,35.6,169,86,126,Prediabetes,False,1200
2,2025-12-09 03:31:46.404501,Ginecologia,Dra. Castillo,P2584,39,F,Este,34.5,143,66,93,Diabetes y Hipertension,False,900
3,2025-12-09 03:31:46.404501,Pediatria,Dr. Jimenez,P2291,69,M,Centro,37.9,158,94,101,Diabetes y Hipertension,False,600
4,2025-12-09 03:31:46.404501,Ginecologia,Dra. Castillo,P4814,54,F,Norte,27.1,129,109,144,Ninguno,False,900
5,2025-12-09 03:31:46.404501,Endocrinologia,Dr. Flores,P5374,58,F,Norte,26.0,120,83,160,Diabetes,False,1100
6,2025-12-09 03:31:46.404501,Pediatria,Dra. Morales,P4752,28,F,Norte,16.5,171,74,153,Ninguno,False,600


## 2. KPIs Operativos

In [ ]:
df = pd.read_csv("data/consultas_clinica.csv", parse_dates=["fecha"])

hoy        = df["fecha"].max()
hace_30    = hoy - timedelta(days=30)
hace_60    = hoy - timedelta(days=60)

df_mes     = df[df["fecha"] >= hace_30].copy()
df_mes_ant = df[(df["fecha"] >= hace_60) & (df["fecha"] < hace_30)].copy()

total_consultas  = len(df_mes)
consultas_ant    = len(df_mes_ant)
delta_consultas  = ((total_consultas - consultas_ant) / consultas_ant * 100) if consultas_ant > 0 else 0

ingresos_total   = df_mes["ingreso"].sum()
ingresos_ant     = df_mes_ant["ingreso"].sum()
delta_ingresos   = ((ingresos_total - ingresos_ant) / ingresos_ant * 100) if ingresos_ant > 0 else 0

pacientes_nuevos    = df_mes["es_nuevo"].sum()
pacientes_recurrentes = total_consultas - pacientes_nuevos
pacientes_unicos    = df_mes["paciente_id"].nunique()
ticket_prom         = ingresos_total / total_consultas

print(f"Periodo: {hace_30.date()} -> {hoy.date()}")
print(f"Consultas totales  : {total_consultas:,}  ({delta_consultas:+.1f}% vs mes anterior)")
print(f"Ingresos totales   : ${ingresos_total:,.0f}  ({delta_ingresos:+.1f}% vs mes anterior)")
print(f"Pacientes nuevos   : {pacientes_nuevos:,}")
print(f"Pacientes recurrentes: {pacientes_recurrentes:,}")
print(f"Ticket promedio    : ${ticket_prom:,.0f}")

Periodo: 2026-05-07 -> 2026-06-06
Consultas totales  : 510  (+6.0% vs mes anterior)
Ingresos totales   : $422,800  (+6.6% vs mes anterior)
Pacientes nuevos   : 129
Pacientes recurrentes: 381
Ticket promedio    : $829


## 3. Graficas operativas

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

BG_CARD = "#1e2130"
COLOR_1 = "#4ecca3"
COLOR_2 = "#f7b731"
COLOR_3 = "#ff6b6b"
COLOR_4 = "#a29bfe"
COLORS  = [COLOR_1, COLOR_2, COLOR_3, COLOR_4, "#74b9ff", "#fd79a8"]

plt.rcParams.update({
    "figure.facecolor": BG_CARD, "axes.facecolor": BG_CARD,
    "axes.edgecolor": "#2d3250", "axes.labelcolor": "#8892b0",
    "xtick.color": "#8892b0", "ytick.color": "#8892b0",
    "text.color": "#ccd6f6", "grid.color": "#2d3250", "grid.alpha": 0.5,
})

os.makedirs("charts", exist_ok=True)

# 1. Tendencia de consultas diarias
cons_dia = df_mes.groupby("fecha").size().reset_index(name="consultas")
cons_dia["mm7"] = cons_dia["consultas"].rolling(7, min_periods=1).mean()
fig, ax = plt.subplots(figsize=(11, 3.5))
ax.fill_between(cons_dia["fecha"], cons_dia["consultas"], alpha=0.15, color=COLOR_1)
ax.plot(cons_dia["fecha"], cons_dia["consultas"], color=COLOR_1, linewidth=1.5, label="Consultas diarias")
ax.plot(cons_dia["fecha"], cons_dia["mm7"], color=COLOR_2, linewidth=2, linestyle="--", label="Media movil 7d")
ax.set_title("Consultas Diarias", color="#ccd6f6", fontsize=12, pad=10)
ax.legend(facecolor=BG_CARD, edgecolor="#2d3250", labelcolor="#ccd6f6")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("charts/tendencia.png", dpi=150, bbox_inches="tight", facecolor=BG_CARD)
plt.show()

# 2. Ingresos por especialidad
ing_esp = df_mes.groupby("especialidad")["ingreso"].sum().sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(11, 3.5))
bars = ax.barh(ing_esp.index, ing_esp.values, color=COLORS[:len(ing_esp)], alpha=0.85, height=0.6)
for bar, val in zip(bars, ing_esp.values):
    ax.text(val + ing_esp.values.max()*0.01, bar.get_y()+bar.get_height()/2,
            f"${val:,.0f}", va="center", color="#ccd6f6", fontsize=8)
ax.set_title("Ingresos por Especialidad", color="#ccd6f6", fontsize=12, pad=10)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f"${x/1000:.0f}k"))
ax.grid(True, axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig("charts/ingresos_especialidad.png", dpi=150, bbox_inches="tight", facecolor=BG_CARD)
plt.show()

# 3. Ocupacion por medico (top 8)
ocu_med = df_mes.groupby("medico").size().sort_values(ascending=True).tail(8)
fig, ax = plt.subplots(figsize=(11, 3.5))
ax.barh(ocu_med.index, ocu_med.values, color=COLOR_4, alpha=0.85, height=0.6)
for i, val in enumerate(ocu_med.values):
    ax.text(val + 0.5, i, str(val), va="center", color="#ccd6f6", fontsize=9)
ax.set_title("Consultas por Medico (Top 8)", color="#ccd6f6", fontsize=12, pad=10)
ax.grid(True, axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig("charts/ocupacion_medico.png", dpi=150, bbox_inches="tight", facecolor=BG_CARD)
plt.show()

# 4. Nuevos vs Recurrentes
fig, ax = plt.subplots(figsize=(5, 3.5))
vals  = [pacientes_nuevos, pacientes_recurrentes]
labs  = [f"Nuevos\n{pacientes_nuevos}", f"Recurrentes\n{pacientes_recurrentes}"]
wedges, texts, autotexts = ax.pie(vals, labels=labs, autopct="%1.1f%%",
    colors=[COLOR_1, COLOR_2], pctdistance=0.75, startangle=90,
    wedgeprops=dict(width=0.5, edgecolor=BG_CARD, linewidth=2))
for t in texts:     t.set_color("#ccd6f6"); t.set_fontsize(9)
for t in autotexts: t.set_color("#0f1117"); t.set_fontsize(8); t.set_fontweight("bold")
ax.set_title("Nuevos vs Recurrentes", color="#ccd6f6", fontsize=12, pad=10)
plt.tight_layout()
plt.savefig("charts/nuevos_recurrentes.png", dpi=150, bbox_inches="tight", facecolor=BG_CARD)
plt.show()

print("Graficas operativas generadas")

Graficas operativas generadas


## 4. Modelo de Propension a Enfermedades Cronicas

> Usamos **Random Forest** para clasificar pacientes segun su riesgo de desarrollar
> diabetes, hipertension u obesidad, basado en su perfil clinico y demografico.
>
> Las variables mas importantes son: IMC, glucosa, presion arterial, edad y diagnosticos previos.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt

# Preparar features
df_ml = df.copy()

# Crear variable objetivo: riesgo de enfermedad cronica
def clasificar_riesgo(row):
    score = 0
    if row["imc"] >= 30:           score += 2
    elif row["imc"] >= 25:         score += 1
    if row["glucosa"] >= 126:      score += 3
    elif row["glucosa"] >= 100:    score += 1
    if row["presion_sistolica"] >= 140: score += 3
    elif row["presion_sistolica"] >= 130: score += 1
    if row["edad"] >= 50:          score += 1
    if "Diabetes" in str(row["diagnostico_previo"]):     score += 3
    if "Hipertension" in str(row["diagnostico_previo"]): score += 3
    if "Obesidad" in str(row["diagnostico_previo"]):     score += 2
    if score >= 6:   return "Alto"
    elif score >= 3: return "Medio"
    else:            return "Bajo"

df_ml["riesgo"] = df_ml.apply(clasificar_riesgo, axis=1)

# Encoders
le_genero = LabelEncoder()
le_zona   = LabelEncoder()
le_diag   = LabelEncoder()
df_ml["genero_enc"]  = le_genero.fit_transform(df_ml["genero"])
df_ml["zona_enc"]    = le_zona.fit_transform(df_ml["zona"])
df_ml["diag_enc"]    = le_diag.fit_transform(df_ml["diagnostico_previo"])

features = ["edad","imc","presion_sistolica","presion_diastolica","glucosa",
            "genero_enc","zona_enc","diag_enc"]
X = df_ml[features]
y = df_ml["riesgo"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

modelo = RandomForestClassifier(n_estimators=100, random_state=42, class_weight="balanced")
modelo.fit(X_train, y_train)

print("Desempeno del modelo:")
print(classification_report(y_test, modelo.predict(X_test)))

# Prediccion en todos los pacientes
df_ml["riesgo_pred"] = modelo.predict(X)
df_ml["prob_alto"]   = modelo.predict_proba(X)[:, list(modelo.classes_).index("Alto")]

print("\nDistribucion de riesgo predicho:")
print(df_ml["riesgo_pred"].value_counts())

Desempeno del modelo:
              precision    recall  f1-score   support

        Alto       0.95      0.97      0.96       376
        Bajo       0.93      0.65      0.77        43
       Medio       0.85      0.88      0.86       180

    accuracy                           0.92       599
   macro avg       0.91      0.83      0.86       599
weighted avg       0.92      0.92      0.91       599


Distribucion de riesgo predicho:
riesgo_pred
Alto     1874
Medio     922
Bajo      195
Name: count, dtype: int64


## 5. Visualizacion del modelo

In [ ]:
# 1. Feature importance
importances = pd.Series(modelo.feature_importances_, index=features).sort_values(ascending=True)
feat_names  = {
    "edad":"Edad", "imc":"IMC", "presion_sistolica":"Presion Sistolica",
    "presion_diastolica":"Presion Diastolica", "glucosa":"Glucosa",
    "genero_enc":"Genero", "zona_enc":"Zona", "diag_enc":"Diagnostico Previo"
}
importances.index = [feat_names.get(i,i) for i in importances.index]

fig, ax = plt.subplots(figsize=(9, 3.5))
colors_fi = [COLOR_1 if v >= importances.values[-3:].min() else COLOR_4 for v in importances.values]
ax.barh(importances.index, importances.values, color=colors_fi, alpha=0.85, height=0.6)
ax.set_title("Importancia de Variables — Modelo de Riesgo", color="#ccd6f6", fontsize=12, pad=10)
ax.set_xlabel("Importancia relativa")
ax.grid(True, axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig("charts/feature_importance.png", dpi=150, bbox_inches="tight", facecolor=BG_CARD)
plt.show()

# 2. Distribucion de riesgo
riesgo_dist = df_ml["riesgo_pred"].value_counts()
fig, ax = plt.subplots(figsize=(5.5, 3.5))
color_riesgo = {"Alto": COLOR_3, "Medio": COLOR_2, "Bajo": COLOR_1}
bars = ax.bar(riesgo_dist.index, riesgo_dist.values,
              color=[color_riesgo.get(r, COLOR_4) for r in riesgo_dist.index],
              alpha=0.85, width=0.5)
for bar, val in zip(bars, riesgo_dist.values):
    ax.text(bar.get_x()+bar.get_width()/2, val+1, str(val),
            ha="center", color="#ccd6f6", fontsize=10, fontweight="bold")
ax.set_title("Distribucion de Pacientes por Nivel de Riesgo", color="#ccd6f6", fontsize=12, pad=10)
ax.set_ylabel("Numero de pacientes")
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("charts/distribucion_riesgo.png", dpi=150, bbox_inches="tight", facecolor=BG_CARD)
plt.show()

# 3. Riesgo por especialidad
riesgo_esp = df_ml.groupby(["especialidad","riesgo_pred"]).size().unstack(fill_value=0)
fig, ax = plt.subplots(figsize=(11, 3.5))
x = np.arange(len(riesgo_esp))
w = 0.25
for i, (nivel, color) in enumerate(color_riesgo.items()):
    if nivel in riesgo_esp.columns:
        ax.bar(x + i*w, riesgo_esp[nivel], width=w, label=nivel, color=color, alpha=0.85)
ax.set_xticks(x + w)
ax.set_xticklabels(riesgo_esp.index, rotation=20, ha="right", fontsize=8)
ax.set_title("Nivel de Riesgo por Especialidad", color="#ccd6f6", fontsize=12, pad=10)
ax.legend(facecolor=BG_CARD, edgecolor="#2d3250", labelcolor="#ccd6f6")
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("charts/riesgo_especialidad.png", dpi=150, bbox_inches="tight", facecolor=BG_CARD)
plt.show()

print("Graficas del modelo generadas")

Graficas del modelo generadas


## 6. Top pacientes de alto riesgo

In [ ]:
# Top 10 pacientes con mayor probabilidad de riesgo alto
top_riesgo = (df_ml[df_ml["riesgo_pred"]=="Alto"]
    .sort_values("prob_alto", ascending=False)
    .drop_duplicates("paciente_id")
    .head(10)[["paciente_id","edad","genero","imc","glucosa",
               "presion_sistolica","diagnostico_previo","prob_alto"]]
    .copy())

top_riesgo["prob_alto"] = (top_riesgo["prob_alto"]*100).round(1).astype(str) + "%"
top_riesgo.columns = ["Paciente","Edad","Genero","IMC","Glucosa","Presion Sist.","Diag. Previo","Prob. Riesgo"]
print("Top 10 pacientes con mayor riesgo:")
print(top_riesgo.to_string(index=False))

Top 10 pacientes con mayor riesgo:
Paciente  Edad Genero  IMC  Glucosa  Presion Sist.            Diag. Previo Prob. Riesgo
   P1104    69      F 23.5      197            160            Hipertension       100.0%
   P3734    23      F 26.5      165            146            Hipertension       100.0%
   P2066    69      F 23.4      196            163                Diabetes       100.0%
   P3854    49      F 33.0      133            173                Diabetes       100.0%
   P2458    72      F 33.0      113            165                 Ninguno       100.0%
   P1500    23      F 35.4      144            168               Sobrepeso       100.0%
   P7938    60      M 33.0      139            163                 Ninguno       100.0%
   P8987    45      M 26.6      133            170                 Ninguno       100.0%
   P5342    51      M 29.7      170            179 Diabetes y Hipertension       100.0%
   P8328    65      F 30.1      149            119            Hipertension       100.

## 7. Generacion del PDF

> Reporte de 3 paginas:
> - **Pagina 1:** KPIs + tendencia de consultas + nuevos vs recurrentes
> - **Pagina 2:** Ingresos por especialidad + ocupacion medicos
> - **Pagina 3:** Modelo ML — importancia de variables + distribucion de riesgo + tabla de alto riesgo


In [ ]:
from fpdf import FPDF

EMPRESA   = "Clinica Privada Demo"
FONT_NAME = "Helvetica"

try:
    FONT_R = "/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf"
    FONT_B = "/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf"
    USE_LIB = os.path.exists(FONT_R)
    if USE_LIB: FONT_NAME = "Liberation"
except:
    USE_LIB = False

class ReportePDF(FPDF):

    def header(self):
        if USE_LIB:
            self.add_font("Liberation","",FONT_R)
            self.add_font("Liberation","B",FONT_B)
        self.set_fill_color(30,33,48)
        self.rect(0,0,210,18,"F")
        self.set_font(FONT_NAME,"B",13)
        self.set_text_color(78,204,163)
        self.set_xy(10,4)
        self.cell(0,10,EMPRESA)
        self.set_font(FONT_NAME,"",8)
        self.set_text_color(136,146,176)
        self.set_xy(10,11)
        self.cell(0,5,f"Reporte Mensual  |  {hace_30.strftime('%d %b')} - {hoy.strftime('%d %b %Y')}")
        self.ln(10)

    def footer(self):
        self.set_y(-12)
        self.set_fill_color(30,33,48)
        self.rect(0,self.get_y(),210,12,"F")
        self.set_font(FONT_NAME,"",7)
        self.set_text_color(77,85,102)
        self.cell(0,10,f"Generado el {datetime.today().strftime('%d/%m/%Y %H:%M')}  |  Luis Fernando Castillo Garcia  |  github.com/Luisinho-31",align="C")

    def kpi_card(self,x,y,w,h,label,value,delta=None,pos=True):
        self.set_fill_color(37,40,55); self.set_draw_color(45,50,80)
        self.rect(x,y,w,h,"FD")
        self.set_font(FONT_NAME,"B",14); self.set_text_color(78,204,163)
        self.set_xy(x,y+3); self.cell(w,8,value,align="C")
        self.set_font(FONT_NAME,"",7); self.set_text_color(136,146,176)
        self.set_xy(x,y+11); self.cell(w,5,label.upper(),align="C")
        if delta is not None:
            self.set_text_color(*(78,204,163) if pos else (255,107,107))
            self.set_font(FONT_NAME,"",7); self.set_xy(x,y+17)
            self.cell(w,4,f"{'+'if pos else '-'}{abs(delta):.1f}% vs ant.",align="C")

    def section_title(self,title):
        self.set_font(FONT_NAME,"B",10); self.set_text_color(78,204,163)
        self.set_fill_color(30,33,48)
        self.cell(0,7,f"  {title}",new_x="LMARGIN",new_y="NEXT",fill=True)
        self.ln(2)

pdf = ReportePDF()
pdf.set_auto_page_break(auto=True,margin=14)

# ── PAGINA 1: KPIs + Tendencia + Nuevos vs Recurrentes ────────────────────────
pdf.add_page()
pdf.set_xy(10,22)
pdf.set_font(FONT_NAME,"B",9); pdf.set_text_color(136,146,176)
pdf.cell(0,5,"RESUMEN EJECUTIVO",new_x="LMARGIN",new_y="NEXT"); pdf.ln(1)

y_kpi = 29; cw = 37; ch = 25
kpis = [
    ("Consultas",    str(total_consultas),   delta_consultas, delta_consultas>=0),
    ("Ingresos",     f"${ingresos_total/1e3:.0f}k", delta_ingresos, delta_ingresos>=0),
    ("Nuevos",       str(pacientes_nuevos),  None, True),
    ("Recurrentes",  str(pacientes_recurrentes), None, True),
    ("Ticket Prom.", f"${ticket_prom:,.0f}", None, True),
]
for i,(label,value,delta,pos) in enumerate(kpis):
    pdf.kpi_card(10+i*40,y_kpi,cw,ch,label,value,delta,pos)

y_tend = y_kpi+ch+8
pdf.set_xy(10,y_tend); pdf.section_title("CONSULTAS DIARIAS")
pdf.image("charts/tendencia.png",x=10,y=y_tend+8,w=130)
pdf.image("charts/nuevos_recurrentes.png",x=143,y=y_tend+8,w=57)

# ── PAGINA 2: Ingresos + Ocupacion ────────────────────────────────────────────
pdf.add_page()
pdf.set_xy(10,22); pdf.section_title("INGRESOS POR ESPECIALIDAD")
pdf.image("charts/ingresos_especialidad.png",x=10,y=30,w=190)
pdf.set_xy(10,88); pdf.section_title("CONSULTAS POR MEDICO")
pdf.image("charts/ocupacion_medico.png",x=10,y=96,w=190)

# ── PAGINA 3: Modelo ML ────────────────────────────────────────────────────────
pdf.add_page()
pdf.set_xy(10,22); pdf.section_title("MODELO DE PROPENSION - ENFERMEDADES CRONICAS")

pdf.set_font(FONT_NAME,"",8); pdf.set_text_color(136,146,176)
pdf.set_xy(10,30)
pdf.multi_cell(190,5,"Modelo Random Forest entrenado con variables clinicas y demograficas. Clasifica pacientes en tres niveles de riesgo: Alto, Medio y Bajo para enfermedades cronicas (diabetes, hipertension, obesidad).")
pdf.ln(2)

pdf.image("charts/feature_importance.png",x=10,y=50,w=110)
pdf.image("charts/distribucion_riesgo.png",x=123,y=50,w=77)

# Tabla top pacientes alto riesgo
y_tab = 108
pdf.set_xy(10,y_tab); pdf.section_title("TOP 10 PACIENTES - RIESGO ALTO")

headers = ["Paciente","Edad","Genero","IMC","Glucosa","P.Sist.","Diag. Previo","Prob."]
col_w   = [25,15,15,15,18,15,62,20]
pdf.set_fill_color(30,33,48); pdf.set_font(FONT_NAME,"B",7); pdf.set_text_color(78,204,163)
x_tab = 10
for h,w in zip(headers,col_w):
    pdf.set_xy(x_tab,y_tab+8); pdf.cell(w,6,h,fill=True,align="C"); x_tab+=w

top10 = (df_ml[df_ml["riesgo_pred"]=="Alto"]
    .sort_values("prob_alto",ascending=False)
    .drop_duplicates("paciente_id").head(10))

for idx,(_, row) in enumerate(top10.iterrows()):
    y_row = y_tab+15+idx*7
    pdf.set_fill_color(61,21,21)
    vals = [row["paciente_id"],str(int(row["edad"])),row["genero"],
            str(round(row["imc"],1)),str(int(row["glucosa"])),
            str(int(row["presion_sistolica"])),
            str(row["diagnostico_previo"])[:30],
            f"{row['prob_alto']*100:.1f}%"]
    x_tab=10
    for v,w in zip(vals,col_w):
        pdf.set_xy(x_tab,y_row); pdf.set_font(FONT_NAME,"",7)
        pdf.set_text_color(255,107,107); pdf.cell(w,6,str(v),fill=True,align="C"); x_tab+=w

os.makedirs("output",exist_ok=True)
fecha_str = datetime.today().strftime("%Y%m%d")
out_path  = f"output/reporte_clinica_{fecha_str}.pdf"
pdf.output(out_path)
print(f"Reporte generado: {out_path}")

Reporte generado: output/reporte_clinica_20260606.pdf


## 8. Descargar PDF desde Colab

In [ ]:
from google.colab import files
files.download(out_path)
print("Descarga iniciada")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Descarga iniciada


---
## Conclusiones

Este reporte combina analisis operativo con inteligencia clinica:

| Modulo | Valor para la clinica |
|---|---|
| KPIs operativos | Visibilidad inmediata de consultas e ingresos |
| Tendencia | Detectar caidas o picos de demanda |
| Ingresos por especialidad | Identificar especialidades mas y menos rentables |
| Ocupacion medicos | Balancear carga de trabajo |
| Modelo ML | Anticipar que pacientes necesitan seguimiento preventivo |
| Top riesgo alto | Lista accionable para el equipo medico |

### Adaptaciones para un caso real
- Conectar a sistema de gestion de la clinica o Excel del cliente
- Agregar modelos por enfermedad especifica (diabetes tipo 2, hipertension)
- Envio automatico del reporte al director medico cada mes
- Dashboard interactivo complementario

---
**Luis Fernando Castillo Garcia**  
[LinkedIn](https://www.linkedin.com/in/luis-fernando-castillo-garcia) | [GitHub](https://github.com/Luisinho-31)
